<a href="https://colab.research.google.com/github/yasminasouki/DataAnalysis/blob/dev/04_gradcam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np, pandas as pd, tensorflow as tf, os
from tensorflow import keras
from tensorflow.keras.applications.densenet import preprocess_input

PROJ = '/content/drive/MyDrive/pneumonia-project'
IN   = f'{PROJ}/results_03'
OUT  = f'{PROJ}/results_04'
os.makedirs(f'{OUT}/figures', exist_ok=True)

# trained model from notebook 03
model = keras.models.load_model(f'{IN}/densenet121_best.keras')

# images + split (same order)
split = pd.read_csv(f'{PROJ}/split.csv')
images = np.load(f'{PROJ}/images_224.npz', allow_pickle=True)['images']
te = split['split'].values == 'test'
X_te = images[te]

# Joy's test predictions (same order as X_te)
pred = pd.read_csv(f'{IN}/test_predictions.csv')

print("Test images:", X_te.shape, "| predictions:", pred.shape)
print(pred.columns.tolist())
print([l.name for l in model.layers])

Mounted at /content/drive
Test images: (1869, 224, 224) | predictions: (1869, 13)
['patientId', 'age', 'sex', 'view', 'height_px', 'width_px', 'class', 'nih_image', 'real_patient_id', 'split', 'target', 'prob_base', 'prob_cnn']
['input_layer_2', 'densenet121', 'global_average_pooling2d', 'dropout', 'dense']


In [2]:
def prep(imgs):
    x = imgs.astype('float32')[..., None]
    x = np.repeat(x, 3, axis=-1)          # gray -> 3 channels, like in notebook 03
    return preprocess_input(x)

check = model.predict(prep(X_te[:8]), verbose=0).ravel()
print("Now:     ", np.round(check, 3))
print("Notebook 03:", np.round(pred['prob_cnn'].values[:8], 3))

Now:      [0.966 0.713 0.694 0.818 0.896 0.5   0.899 0.821]
Notebook 03: [0.966 0.713 0.694 0.818 0.896 0.5   0.899 0.821]
